In [2]:
# ─── CELL 1: IMPORTS ───
import os
import re
import math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import mean_absolute_error, r2_score

In [3]:
# ─── CELL 2: LOAD FILES ───
dataset_path = "/kaggle/input/datasets/arujatiwary/nasa-li-ion-dataset/ECM_processed_cycles"  # ← update this path
cycle_files = sorted([
    os.path.join(dataset_path, f)
    for f in os.listdir(dataset_path)
    if f.startswith("cycle_")
])
print("Total cycles:", len(cycle_files))

Total cycles: 341


In [4]:
# ─── CELL 3: INITIAL CAPACITY ───
dt = 1.0
Q_used_all_cycles = []
for file in cycle_files:
    df = pd.read_csv(file)
    I = df['Current_measured'].fillna(0).values
    I_discharge = np.where(I < 0, -I, 0)
    Q_cycle = np.sum(I_discharge * dt) / 3600
    Q_used_all_cycles.append(Q_cycle)
initial_capacity = max(Q_used_all_cycles)
print("Initial capacity:", initial_capacity)

Initial capacity: 1.7059162227905784


In [5]:
# ─── CELL 4: FILTER DISCHARGE CYCLES ───
discharge_cycle_files = []
for file in cycle_files:
    df = pd.read_csv(file)
    if df['SOC_ECM'].isna().all():
        continue
    I = df['Current_measured'].fillna(0).values
    I_discharge = np.where(I < 0, -I, 0)
    Q = np.sum(I_discharge * dt) / 3600
    if Q < 0.01:
        continue
    discharge_cycle_files.append(file)
print(f"Clean discharge cycles: {len(discharge_cycle_files)} / {len(cycle_files)}")

Clean discharge cycles: 171 / 341


In [6]:
# ─── CELL 5: SPLIT ───
split       = int(0.8 * len(discharge_cycle_files))
train_files = discharge_cycle_files[:split]
val_files   = discharge_cycle_files[split:]
print(f"Train: {len(train_files)}  Val: {len(val_files)}")

Train: 136  Val: 35


In [7]:
# ─── CELL 6: GLOBAL STATS ───
def compute_global_stats(files):
    num_cols = [
        'Voltage_measured', 'Current_measured', 'dV_dt', 'dT_dt',
        'V_RC_masked', 'V_ECM_masked', 'power', 'Temperature_measured'
    ]
    all_data = []
    for file in files:
        df = pd.read_csv(file)
        df = df.ffill().fillna(0)
        if 'mode_flag' not in df.columns:
            df['mode_flag'] = (df['Current_measured'] < 0).astype(int)
        df['V_RC_masked']  = df['V_RC']  * df['mode_flag']
        df['V_ECM_masked'] = df['V_ECM'] * df['mode_flag']
        df['power']        = df['Voltage_measured'] * df['Current_measured']
        all_data.append(df[num_cols])
    all_data = pd.concat(all_data)
    return all_data.mean(), all_data.std() + 1e-8

global_mean, global_std = compute_global_stats(train_files)
print("Global stats computed.")

Global stats computed.


In [8]:
# ─── CELL 7: POSITIONAL ENCODING ───
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=500):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer("pe", pe)

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

In [9]:
# ─── CELL 8: MODEL ───
class BatteryTransformer(nn.Module):
    def __init__(self, input_dim=11, d_model=128, nhead=4, num_layers=2, dropout=0.2):
        super().__init__()
        self.input_proj  = nn.Linear(input_dim, d_model)
        self.pos_encoder = PositionalEncoding(d_model)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead,
            dim_feedforward=256, dropout=dropout,
            batch_first=True
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        decoder_layer = nn.TransformerDecoderLayer(
            d_model=d_model, nhead=nhead,
            dim_feedforward=256, dropout=dropout,
            batch_first=True
        )
        self.decoder = nn.TransformerDecoder(decoder_layer, num_layers=num_layers)

        self.query_embed = nn.Parameter(torch.randn(1, 3, d_model))

        self.soc_mu_head      = nn.Linear(d_model, 1)
        self.soc_logvar_head  = nn.Linear(d_model, 1)
        self.soh_mu_head      = nn.Linear(d_model, 1)
        self.soh_logvar_head  = nn.Linear(d_model, 1)
        self.temp_mu_head     = nn.Linear(d_model, 1)
        self.temp_logvar_head = nn.Linear(d_model, 1)

        for head in [self.soc_logvar_head, self.soh_logvar_head, self.temp_logvar_head]:
            nn.init.zeros_(head.weight)
            nn.init.zeros_(head.bias)

    def forward(self, x):
        batch_size = x.size(0)
        x = self.input_proj(x)
        x = self.pos_encoder(x)
        memory = self.encoder(x)

        queries = self.query_embed.expand(batch_size, -1, -1)
        decoded = self.decoder(queries, memory)

        soc_feat  = decoded[:, 0, :]
        soh_feat  = decoded[:, 1, :]
        temp_feat = decoded[:, 2, :]

        soc  = torch.cat([self.soc_mu_head(soc_feat),
                          self.soc_logvar_head(soc_feat.detach())],  dim=1)
        soh  = torch.cat([self.soh_mu_head(soh_feat),
                          self.soh_logvar_head(soh_feat.detach())],  dim=1)
        temp = torch.cat([self.temp_mu_head(temp_feat),
                          self.temp_logvar_head(temp_feat.detach())], dim=1)
        return soc, soh, temp

In [10]:
# ─── CELL 9: LOAD SAVED MODEL (skipping training) ───
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

model = BatteryTransformer(input_dim=11).to(device)
model.load_state_dict(torch.load(
    "/kaggle/input/models/arujatiwary/multi-headed-transformer/transformers/default/1/best_model.pt",
    map_location=device
))
model.eval()
print("Loaded best_model.pt — training skipped.")

Device: cuda
Loaded best_model.pt — training skipped.


In [11]:
# ─── CELL 10: SAVE GLOBALS FOR PIPELINE ───
import pickle

with open("/kaggle/working/predictor_globals.pkl", "wb") as f:
    pickle.dump({
        "global_mean":      global_mean,
        "global_std":       global_std,
        "initial_capacity": initial_capacity,
    }, f)

print("Saved predictor_globals.pkl to /kaggle/working/")
print(f"  initial_capacity : {initial_capacity:.4f} Ah")
print(f"  global_mean      : {global_mean.to_dict()}")

Saved predictor_globals.pkl to /kaggle/working/
  initial_capacity : 1.7059 Ah
  global_mean      : {'Voltage_measured': 3.3851577433189393, 'Current_measured': -0.7940517359524797, 'dV_dt': -0.0001288056011663135, 'dT_dt': 0.0005301368807838826, 'V_RC_masked': 0.007944968462111577, 'V_ECM_masked': 3.583018552010538, 'power': -2.6403314754683085, 'Temperature_measured': 8.87462549122455}


In [12]:
# Should recover ~24°C (room temperature)
temp_raw = df['Temperature_measured'].iloc[0]
temp_celsius = temp_raw * float(global_std['Temperature_measured']) + float(global_mean['Temperature_measured'])
print(f"Normalised: {temp_raw:.4f}")
print(f"Recovered Celsius: {temp_celsius:.2f} °C")
print(f"In Kelvin: {temp_celsius + 273.15:.2f} K")

Normalised: 6.3840
Recovered Celsius: 22.89 °C
In Kelvin: 296.04 K
